# 建立影像生成應用程式（OpenAI）

大型語言模型不只是產生文字。您也可以從文字描述生成影像。影像作為一種媒介，在醫療科技、建築、旅遊、遊戲開發、行銷等領域都非常有用。在本課程中，我們將使用 OpenAI 平台上的<strong>GPT 影像</strong>模型。

## 學習目標

完成本課程後，您將能夠：

- 解釋什麼是影像生成以及它的應用場景。
- 了解 `gpt-image` 模型家族及其與傳統 DALL·E 模型的差異。
- 建立影像生成應用程式並進行影像編輯。

## 什麼是影像生成？

影像生成模型從文字提示創建影像。現代模型如 `gpt-image` 在訓練期間學習文字與影像的關係，接著逐步將隨機噪聲轉換成與您的提示相符的影像。

兩個知名的影像模型家族是：

- **`gpt-image`（OpenAI）** — 本課程使用的最新一代模型。它支援文字轉影像生成以及影像編輯（利用遮罩進行修補）。
- **Midjourney** — 一個受歡迎的第三方模型，擁有自己的服務與以 Discord 為基礎的工作流程。

> 舊款的 OpenAI 影像模型 — **DALL·E 2** 和 **DALL·E 3** — 皆為傳統版本。DALL·E 3 已停止新部署，且功能如 `create_variation` 僅存於 DALL·E 2。新應用請使用 `gpt-image` 模型。

> **重要:** `gpt-image` 模型回傳的生成影像為 **base64** 格式（`b64_json`），而非 URL。您的程式需將 base64 字串解碼成位元組並保存 — 不存在可下載的影像 URL。


## 建立你的第一個影像生成應用程式

那麼，要建立一個影像生成應用程式需要什麼呢？你需要以下幾個函式庫：

- **python-dotenv**，強烈建議你使用此函式庫，將你的機密儲存在 *.env* 檔案中，並且與程式碼分開。
- **openai**，這是你將用來與 OpenAI API 互動的函式庫。
- **pillow**，用以在 Python 中處理影像。


1. 建立一個 *.env* 檔案，內容如下：

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. 將上述函式庫收集到一個名為 *requirements.txt* 的檔案中，如下所示：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接下來，建立虛擬環境並安裝函式庫：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 對於 Windows，使用以下命令來建立並啟用您的虛擬環境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名為 *app.py* 的檔案中加入以下程式碼：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # 建立 OpenAI 物件（從你的 .env 讀取 OPENAI_API_KEY）
    client = openai.OpenAI()


    try:
        # 使用圖像生成 API 創建圖像
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
            size='1024x1024',
            n=1
        )
        # 設定存放圖像的目錄
        image_dir = os.path.join(os.curdir, 'images')

        # 如果目錄不存在，則創建它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化圖像路徑（注意檔案類型應為 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型返回圖像為 base64（b64_json），而非 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在預設圖像查看器中顯示圖像
        image = Image.open(image_path)
        image.show()

    # 捕捉例外情況
    except openai.BadRequestError as err:
        print(err)

    ```

讓我們說明這段程式碼：

- 首先，我們匯入所需的函式庫，包括 OpenAI 函式庫、dotenv 函式庫、base64 模組和 Pillow 函式庫。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- 接著，我們建立客戶端，該客戶端從您的 ``.env`` 檔案讀取 API 金鑰。

    ```python
    # 建立 OpenAI 物件
    client = openai.OpenAI()
    ```

- 接下來，我們生成圖片：

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` 模型會回傳一個以 **base64** 編碼的字串於 `data[0].b64_json`。我們將其解碼為位元組並寫入檔案 — 沒有下載用的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後，我們打開圖片並用標準的圖片檢視器顯示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 生成圖片的更多細節

讓我們來看看 `images.generate` 的參數：

- **model**，是要使用的圖片模型（例如 `gpt-image-1`）。
- **prompt**，是用來生成圖片的文字提示。這裡是「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」。
- **size**，是生成圖片的尺寸（例如 `1024x1024`、`1536x1024`、`1024x1536`，或 `"auto"`）。
- **n**，是生成圖片的數量。這裡只生成一張。

> 圖片模型不接受 `temperature` 參數—那是用於文字生成的控制。若要增加多樣性，請再次呼叫 API；若要減少多樣性，請使提示更具體。

## 生成圖片的額外功能

你已經看到如何用幾行 Python 程式碼生成圖片。`gpt-image` 模型也可以用來<strong>編輯</strong>現有圖片。透過提供一張圖片、一個選填的<strong>遮罩</strong>（標示需更改的區域）以及提示詞，你可以修改圖片的一部分 — 例如，給我們的兔子加戴一頂帽子。

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編輯結果也會以 base64 格式回傳
edited_image = base64.b64decode(response.data[0].b64_json)
```

基底圖片只包含兔子；最後的圖片則是兔子戴著帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
